# 🛒 모두마켓 신상품 수집 리포트

**목표**: 모두마켓 상품 목록 HTML에서 상품 정보를 수집 → 정제 → 분석하고, 수집 과정의 신뢰성과 윤리를 함께 기록한다.

| 단계 | 내용 |
|---|---|
| 1 | HTML 준비 (상품 8개, 가격 표기 혼재, 결측 1건) |
| 2 | 페이지 유형 판별 (정적 vs 동적) |
| 3 | BeautifulSoup 수집 + 개수 검증 |
| 4 | 가격 정제 + 결측 처리 |
| 5 | 분석 (카테고리별 평균, 최고가 상품) |
| 6 | 윤리 점검 (robots.txt, `time.sleep`) |

In [1]:
import re
import time
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.unicode.east_asian_width", True)

## 1. HTML 준비

실습 예시를 참고해 **상품 8개**짜리 모두마켓 HTML을 직접 만들었다. 실제 웹 데이터처럼 일부러 지저분하게 구성했다.

- 가격 표기 혼재: `45,000원`, ` 79,000 원 `, `₩23,000`, `KRW 129,000`, `12900원`
- 카테고리 앞뒤 공백: `" 패션 "`, `"가전 "`
- 결측 1건: `P104 프리미엄 원두`의 가격 칸이 비어 있음

In [2]:
shop_html = """
<html>
<head><title>모두마켓 - 신상품</title></head>
<body>
<div id="catalog">
  <div class="item" data-pid="P101"><span class="title">게이밍 마우스</span><span class="cat">가전</span><span class="cost">45,000원</span></div>
  <div class="item" data-pid="P102"><span class="title">린넨 원피스</span><span class="cat"> 패션 </span><span class="cost"> 79,000 원 </span></div>
  <div class="item" data-pid="P103"><span class="title">아로마 캔들</span><span class="cat">뷰티</span><span class="cost">₩23,000</span></div>
  <div class="item" data-pid="P104"><span class="title">프리미엄 원두</span><span class="cat">식품</span><span class="cost"></span></div>
  <div class="item" data-pid="P105"><span class="title">요가 매트</span><span class="cat">스포츠</span><span class="cost">38,500원</span></div>
  <div class="item" data-pid="P106"><span class="title">무선 이어폰</span><span class="cat">가전 </span><span class="cost">KRW 129,000</span></div>
  <div class="item" data-pid="P107"><span class="title">데님 자켓</span><span class="cat">패션</span><span class="cost">  ₩ 99,000 </span></div>
  <div class="item" data-pid="P108"><span class="title">수제 그래놀라</span><span class="cat">식품</span><span class="cost">12900원</span></div>
</div>
</body>
</html>
"""
shop_soup = BeautifulSoup(shop_html, "html.parser")
print("HTML 준비 완료 — 상품 카드:", len(shop_soup.select(".item")), "개")

HTML 준비 완료 — 상품 카드: 8 개


## 2. 페이지 유형 판별 (정적 vs 동적)

**판별 기준**: 브라우저가 JavaScript를 실행하기 *전*의 원본 HTML(페이지 소스, `requests.get().text`)에 상품 데이터가 이미 들어 있으면 **정적**, 비어 있고 JS가 나중에 채워 넣으면 **동적**이다.

아래 코드로 원본 소스에서 직접 확인한다.

In [3]:
# 근거 1: 원본 소스 문자열에 상품명이 그대로 들어 있는가?
sample_names = ["게이밍 마우스", "요가 매트", "수제 그래놀라"]
for name in sample_names:
    print(f"'{name}' 소스에 존재:", name in shop_html)

# 근거 2: 데이터를 불러오는 <script>(fetch/XHR 등)가 있는가?
scripts = shop_soup.find_all("script")
print("<script> 태그 수:", len(scripts))

'게이밍 마우스' 소스에 존재: True
'요가 매트' 소스에 존재: True
'수제 그래놀라' 소스에 존재: True
<script> 태그 수: 0


**판별 결과: 정적 페이지** → BeautifulSoup 정적 파싱 선택

- 근거 1: 상품명·카테고리·가격이 원본 HTML 소스에 그대로 존재한다.
- 근거 2: 데이터를 비동기로 불러오는 `<script>`가 하나도 없다.
- 따라서 Selenium/Playwright 같은 브라우저 자동화가 필요 없고, `requests` + BeautifulSoup으로 충분하다.

> 실제 사이트라면 브라우저에서 **'페이지 소스 보기(Ctrl+U)'** 로 데이터가 보이는지, 또는 개발자도구 Network 탭에서 XHR/Fetch로 JSON을 받는지 확인한다.

## 3. 수집 + 개수 검증

`.item` 카드마다 `data-pid`, 상품명, 카테고리, 가격을 꺼낸다. 이 단계에서는 **원본 그대로** 저장하고 정제는 다음 단계에서 한다(원본 보존).

태그가 없을 때 `.select_one()`은 `None`을 돌려주므로, 바로 `.get_text()`를 부르면 에러가 난다. 안전 함수로 감싼다.

In [4]:
def safe_text(card, selector):
    tag = card.select_one(selector)
    return tag.get_text() if tag else None   # 태그 자체가 없으면 None

rows = []
for card in shop_soup.select("#catalog .item"):
    rows.append({
        "pid":       card.get("data-pid"),
        "title_raw": safe_text(card, ".title"),
        "cat_raw":   safe_text(card, ".cat"),
        "price_raw": safe_text(card, ".cost"),
    })

raw_df = pd.DataFrame(rows)
raw_df

,pid,title_raw,cat_raw,price_raw
0,P101,게이밍 마우스,가전,"45,000원"
1,P102,린넨 원피스,패션,"79,000 원"
2,P103,아로마 캔들,뷰티,"₩23,000"
3,P104,프리미엄 원두,식품,
4,P105,요가 매트,스포츠,"38,500원"
5,P106,무선 이어폰,가전,"KRW 129,000"
6,P107,데님 자켓,패션,"₩ 99,000"
7,P108,수제 그래놀라,식품,12900원


### 조용한 실패(silent failure) 방지 검증

선택자를 잘못 쓰면 에러 없이 0개나 일부만 수집되는 경우가 있다. **서로 다른 방법으로 센 개수**와 대조해 수집 누락이 없는지 확인한다.

In [5]:
expected_by_regex = len(re.findall(r'class="item"', shop_html))   # 원본 문자열에서 직접 센 수
expected_by_pid   = len(shop_soup.find_all(attrs={"data-pid": True}))
n_items           = len(raw_df)

print("정규식으로 센 카드 수 :", expected_by_regex)
print("data-pid 속성 수      :", expected_by_pid)
print("실제 수집 행 수       :", n_items)

assert n_items == expected_by_regex == expected_by_pid, "⚠️ 수집 개수 불일치 — 선택자를 확인하세요"
assert raw_df["pid"].is_unique, "⚠️ 중복 상품 ID 존재"
assert raw_df["title_raw"].notna().all(), "⚠️ 상품명 누락 존재"
print("✅ 검증 통과: 누락·중복 없이", n_items, "개 수집")

정규식으로 센 카드 수 : 8
data-pid 속성 수      : 8
실제 수집 행 수       : 8
✅ 검증 통과: 누락·중복 없이 8 개 수집


## 4. 정제 + 결측 처리

**가격 정제 규칙**
1. `₩`, `원`, `KRW`, 쉼표, 공백 등 **숫자가 아닌 문자를 모두 제거**한다.
2. 남은 문자열이 비어 있으면 결측(`NaN`)으로, 아니면 정수로 변환한다.

카테고리·상품명은 앞뒤 공백을 `strip()`으로 제거한다. (`"가전 "`과 `"가전"`이 다른 그룹으로 집계되는 것을 막기 위함)

In [6]:
def clean_price(text):
    if text is None:
        return pd.NA
    digits = re.sub(r"[^0-9]", "", text)   # 숫자만 남김
    return int(digits) if digits else pd.NA

df = raw_df.copy()
df["title"] = df["title_raw"].str.strip()
df["category"] = df["cat_raw"].str.strip()
df["price"] = df["price_raw"].apply(clean_price).astype("Int64")

# 정제 전/후 비교
df[["pid", "title", "cat_raw", "category", "price_raw", "price"]]

,pid,title,cat_raw,category,price_raw,price
0,P101,게이밍 마우스,가전,가전,"45,000원",45000
1,P102,린넨 원피스,패션,패션,"79,000 원",79000
2,P103,아로마 캔들,뷰티,뷰티,"₩23,000",23000
3,P104,프리미엄 원두,식품,식품,,<NA>
4,P105,요가 매트,스포츠,스포츠,"38,500원",38500
5,P106,무선 이어폰,가전,가전,"KRW 129,000",129000
6,P107,데님 자켓,패션,패션,"₩ 99,000",99000
7,P108,수제 그래놀라,식품,식품,12900원,12900


In [7]:
# 정제 결과 검증
print("정제 전 카테고리:", sorted(raw_df["cat_raw"].unique()))
print("정제 후 카테고리:", sorted(df["category"].unique()))

n_missing = int(df["price"].isna().sum())
n_priced  = int(df["price"].notna().sum())
print(f"\n가격 확보 {n_priced}개 / 결측 {n_missing}개")
df[df["price"].isna()][["pid", "title", "category", "price_raw"]]

정제 전 카테고리: [' 패션 ', '가전', '가전 ', '뷰티', '스포츠', '식품', '패션']
정제 후 카테고리: ['가전', '뷰티', '스포츠', '식품', '패션']

가격 확보 7개 / 결측 1개


,pid,title,category,price_raw
3,P104,프리미엄 원두,식품,


### 결측 처리 판단: **집계에서 제외 (행은 보존)**

| 선택지 | 판단 |
|---|---|
| 0원으로 채우기 | ❌ 평균을 왜곡하고 '무료 상품'처럼 보이게 함 |
| 평균/중앙값으로 채우기 | ❌ 상품마다 가격대가 달라 근거 없는 값이 됨 (식품 카테고리 2개뿐) |
| 행 삭제 | △ '상품이 존재한다'는 사실 자체가 사라짐 |
| **NaN 유지 → 가격 집계에서만 제외** | ✅ 상품 목록에는 남기고, 평균·최고가 계산에는 영향 없음 |

근거: 결측은 1건(전체의 약 12%)이고 원인이 '가격 미등록(빈 태그)'으로 확인되어 임의 값을 넣을 근거가 없다. pandas의 `mean()`, `max()`는 기본적으로 NaN을 건너뛰므로 **집계 시 자동 제외**된다.

In [8]:
clean_df = df[["pid", "title", "category", "price"]]
clean_df

,pid,title,category,price
0,P101,게이밍 마우스,가전,45000
1,P102,린넨 원피스,패션,79000
2,P103,아로마 캔들,뷰티,23000
3,P104,프리미엄 원두,식품,<NA>
4,P105,요가 매트,스포츠,38500
5,P106,무선 이어폰,가전,129000
6,P107,데님 자켓,패션,99000
7,P108,수제 그래놀라,식품,12900


## 5. 분석

### 5-1. 카테고리별 상품 수 · 평균 가격

In [9]:
cat_summary = (
    clean_df.groupby("category")
    .agg(상품수=("pid", "count"),
         가격확보=("price", "count"),
         평균가격=("price", "mean"))
    .sort_values("평균가격", ascending=False)
)
cat_summary["평균가격"] = cat_summary["평균가격"].round(0).astype("Int64")
cat_summary

,상품수,가격확보,평균가격
category,,,
패션,2,2,89000
가전,2,2,87000
스포츠,1,1,38500
뷰티,1,1,23000
식품,2,1,12900


### 5-2. 최고가 / 최저가 상품, 전체 평균

In [10]:
priced = clean_df.dropna(subset=["price"])

top_item = priced.loc[priced["price"].idxmax()]
low_item = priced.loc[priced["price"].idxmin()]
avg_price = round(float(priced["price"].mean()))
top_cat = cat_summary.index[0]

print(f"최고가 상품   : {top_item['title']} ({top_item['category']}) — {top_item['price']:,}원")
print(f"최저가 상품   : {low_item['title']} ({low_item['category']}) — {low_item['price']:,}원")
print(f"전체 평균 가격: 약 {avg_price:,}원 (가격 확보 {len(priced)}개 기준)")
print(f"평균가 최고 카테고리: {top_cat}")

최고가 상품   : 무선 이어폰 (가전) — 129,000원
최저가 상품   : 수제 그래놀라 (식품) — 12,900원
전체 평균 가격: 약 60,914원 (가격 확보 7개 기준)
평균가 최고 카테고리: 패션


## 6. 윤리 점검

### 6-1. robots.txt 확인 방법
사이트 루트의 `/robots.txt`에서 수집하려는 경로가 허용되는지 확인한다. 파이썬 내장 `urllib.robotparser`로 자동 확인할 수 있다.

아래는 연습용 샌드박스 사이트(books.toscrape.com)로 패턴을 검증하는 코드다. 네트워크가 막힌 환경에서도 노트북이 멈추지 않도록 예외 처리했다.

In [ ]:
from urllib import robotparser

def check_robots(base_url, path, agent="*"):
    rp = robotparser.RobotFileParser()
    rp.set_url(base_url.rstrip("/") + "/robots.txt")
    try:
        rp.read()
        allowed = rp.can_fetch(agent, base_url.rstrip("/") + path)
        print(f"[{base_url}{path}] 수집 허용 여부: {allowed}")
        return allowed
    except Exception as e:
        print(f"robots.txt를 읽지 못함 ({type(e).__name__}) — 허용 여부 확인 전에는 수집하지 않는다")
        return None

check_robots("https://books.toscrape.com", "/catalogue/page-1.html")

> robots.txt 파일이 **없으면**(404) `robotparser`는 전체 허용으로 해석한다. 그래도 이용약관(ToS)을 함께 확인하는 것이 원칙이다.

### 6-2. 요청 예의: `time.sleep()`
여러 페이지를 요청할 때는 요청 사이에 간격을 두어 서버에 부담을 주지 않는다. 또한 `User-Agent`로 누가 요청하는지 밝힌다.

In [ ]:
import requests

HEADERS = {"User-Agent": "ModuMarket-Study-Bot/1.0 (for study)"}  # HTTP 헤더는 영문(ASCII)만 가능
DELAY_SEC = 1.0   # 요청 간 최소 1초 대기

def polite_get(url):
    """요청 후 반드시 쉬는 예의 바른 GET"""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        return resp.text
    except requests.RequestException as e:
        print("요청 실패:", type(e).__name__)
        return None
    finally:
        time.sleep(DELAY_SEC)

# 샌드박스 2페이지만 시범 수집 (패턴 검증용)
for page in range(1, 3):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    html = polite_get(url)
    if html:
        n = len(BeautifulSoup(html, "html.parser").select("article.product_pod"))
        print(f"page {page}: 책 {n}권 확인")

### 6-3. 윤리 체크리스트
- [x] robots.txt에서 수집 경로 허용 여부 확인
- [x] 요청 간 `time.sleep(1.0)` 적용, `User-Agent` 명시, `timeout` 설정
- [x] 연습 허용 샌드박스(books.toscrape.com)와 직접 만든 HTML만 사용
- [x] 개인정보(이름·연락처·리뷰 작성자 등) 미수집 — 상품명·카테고리·가격만 수집
- [x] 수집 결과는 학습 목적으로만 사용, 재배포하지 않음

In [13]:
# 리포트에 들어갈 숫자 요약 (아래 리포트 칸을 채우는 데 사용)
summary = {
    "n_items": n_items, "n_priced": n_priced, "n_missing": n_missing,
    "avg_price": f"{avg_price:,}", "top_cat": top_cat,
    "top_item": f"{top_item['title']} ({top_item['price']:,}원)",
}
summary

{'n_items': 8,
 'n_priced': 7,
 'n_missing': 1,
 'avg_price': '60,914',
 'top_cat': '패션',
 'top_item': '무선 이어폰 (129,000원)'}

---
# 📋 데이터 수집 리포트

### 1. 수집 개요
- **대상**: 모두마켓 신상품 목록 페이지 (+ 샌드박스 books.toscrape.com에서 수집 패턴 검증)
- **방법**: 직접 작성한 HTML(실제 수집 시 `requests`) → BeautifulSoup 정적 파싱 → DataFrame → 문자열·결측 정제 → 집계
- **페이지 유형 판별**: **정적**. 원본 소스에 상품명·카테고리·가격이 모두 존재하고 데이터를 불러오는 `<script>`가 0개 → BeautifulSoup 선택

### 2. 수집 결과
- 상품 **8개** 수집, 가격 확보 **7개** / 결측 **1개** (P104 프리미엄 원두).
- 전체 평균 가격 약 **60,914원**, 평균가 최고 카테고리는 **패션**(평균 89,000원).
- 최고가 상품은 **무선 이어폰(가전, 129,000원)**, 최저가 상품은 **수제 그래놀라(식품, 12,900원)**.
- 카테고리별 평균: 패션 89,000원 > 가전 87,000원 > 스포츠 38,500원 > 뷰티 23,000원 > 식품 12,900원(1개 기준)

### 3. 데이터 신뢰성
- **검증**: 수집 행 수(8)를 ① 원본 문자열 정규식 카운트(8) ② `data-pid` 속성 수(8)와 대조해 일치 확인 → 조용한 실패 없음. 상품 ID 중복·상품명 누락도 `assert`로 점검.
- **정제**: 가격에서 `₩`, `원`, `KRW`, 쉼표, 공백 등 숫자 외 문자를 제거 후 정수 변환. 카테고리 앞뒤 공백 제거로 7개로 쪼개졌던 카테고리를 5개로 통합(`"가전 "`→`"가전"`, `" 패션 "`→`"패션"`).
- **결측**: 1건(빈 가격 태그)은 임의 값으로 채울 근거가 없어 **NaN으로 두고 가격 집계에서만 제외**, 상품 목록에는 유지.
- **한계**: 식품 카테고리 평균은 가격이 있는 1개 상품 기준이라 대표성이 낮다.

### 4. 윤리 점검
- **robots.txt**: `urllib.robotparser`로 `/robots.txt`를 읽어 `can_fetch()`로 수집 경로 허용 여부를 확인. 확인되지 않으면 수집하지 않는 것을 원칙으로 함.
- **요청 예의**: 요청마다 `time.sleep(1.0)` 적용, `User-Agent` 명시, `timeout=10` 설정, 시범 수집은 2페이지로 제한.
- **범위**: 직접 만든 HTML과 연습 허용 샌드박스만 사용. 개인정보 미수집, 학습 목적으로만 사용.